In [1]:
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-udh6lu7z
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-udh6lu7z
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36592 sha256=9235ae7ebef7d759c8623e7821d01dc1af3860e4cd9a895c850db413d4568178
  Stored in directory: /tmp/pip-ephem-wheel-cache-_n0hjao2/wheels/15/d7/bd/05f5f23b7dcbe70cbc6783b06f12143b0cf1a5da5c7b52dcc5
Successfully built segment_anything


In [2]:
!wget -O model.pth "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"

--2025-02-17 14:49:15--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.167.112.53, 3.167.112.51, 3.167.112.66, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.167.112.53|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2564550879 (2.4G) [binary/octet-stream]
Saving to: ‘model.pth’

model.pth           100%[===================>]   2.39G   111MB/s    in 19s     

2025-02-17 14:49:34 (131 MB/s) - ‘model.pth’ saved [2564550879/2564550879]



In [3]:
from segment_anything import SamPredictor, sam_model_registry, SamAutomaticMaskGenerator
import cv2
import numpy as np
sam = sam_model_registry["vit_h"](checkpoint="/content/model.pth")
predictor = SamPredictor(sam)

/usr/local/lib/python3.11/dist-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


In [4]:
import cv2
image = cv2.imread("/content/original.png")  # Load image
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB

In [ ]:
predictor.set_image(image)
masks, _, _ = predictor.predict(<input_prompts>)

In [5]:
import torch
import numpy as np
import cv2
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# Load the SAM model
sam_checkpoint = "/content/model.pth"  # Ensure this is downloaded
model_type = "vit_h"
device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint).to(device)
mask_generator = SamAutomaticMaskGenerator(sam)

# Load and preprocess the image
image_path = "/content/original.png"  # Replace with your image path
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Generate masks
masks = mask_generator.generate(image_rgb)

# Initialize output image
output = image.copy()
centroids = []

# Loop through each segment mask
for i, mask in enumerate(masks):
    segmentation_mask = mask["segmentation"].astype(np.uint8) * 255  # Convert to 0-255 range

    # Compute centroid using image moments
    M = cv2.moments(segmentation_mask)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])  # X centroid
        cy = int(M["m01"] / M["m00"])  # Y centroid
        centroids.append((cx, cy))

        # Draw centroid on the output image
        cv2.circle(output, (cx, cy), 5, (0, 255, 0), -1)
        cv2.putText(output, f"{i+1}", (cx, cy-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

# Display results
cv2.imwrite("output_with_centroids.jpg", output)
cv2.imshow("Segmented Image with Centroids", output)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Print the number of segments and centroids
print(f"Number of segments: {len(masks)}")
print("Centroids of each segment:", centroids)


/usr/local/lib/python3.11/dist-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


DisabledFunctionError: cv2.imshow() is disabled in Colab, because it causes Jupyter sessions
to crash; see https://github.com/jupyter/notebook/issues/3935.
As a substitution, consider using
  from google.colab.patches import cv2_imshow
